# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [15]:
import os
# from util import retrieve_docs, _retrieve_hub_docs, get_mds
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import requests
from utilkopia import get_urls


load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")

openai = OpenAI()

API key looks good so far


In [ ]:
MODEL_LLAMA = 'llama3.2'
MODEL_GPT = 'gpt-5-nano'
MODEL_DEEPSEEK = 'deepseek-r1:1.5b'
MODEL_MINISTRAL = 'ministral-3:8b'

KOPIA_DOCS_URL = 'https://kopia.io/docs'

In [10]:
lprmt = """
I need to retrieve all the documentation that can be found within a documentation directory
 with the url https://github.com/kopia/kopia/tree/master/site/content/docs.
Therefore I  want urls of all the pages that have '_index.md' as url that reside with the documentation directory
 that a user can find within https://github.com/kopia/kopia/tree/master/site/content/docs, no information from webpages outside this url path.
Exclude pages about Contribution guidelines, Privacy policy and also exclude Release notes.
Explain how I can retrieve the urls by using python and possibly if needed BeautifulSoup.
"""

In [11]:
messages = [
            {"role": "system", "content": """Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""},
            {"role": "user", "content": f"{lprmt}"}
        ]

In [10]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [11]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages
    )

In [12]:
response_gpt = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages
)

In [14]:
display(Markdown(response_gpt.choices[0].message.content))

Here's a practical way to retrieve all _index.md pages that live under the GitHub path site/content/docs, excluding the Contribution guidelines, Privacy policy, and Release notes pages. I’ll show a robust API-based approach (preferred) and a fallback HTML-parsing approach (if you must work with the rendered HTML).

What you’ll achieve
- List all files named _index.md that reside anywhere under site/content/docs in kopia/kopia.
- Exclude pages that correspond to Contribution guidelines, Privacy policy, and Release notes.
- Return the GitHub file URLs (blob URLs) for those _index.md files, which live under the same repo path.

Prerequisites
- Python 3.x
- The requests library (pip install requests)
- Optional: a GitHub personal access token if you need higher rate limits (export GITHUB_TOKEN=your_token)

Option A: API-based approach (recommended)
Why API: reliable, structured, and faster than scraping HTML. Easy to recurse through directories under site/content/docs and filter by _index.md file names.

What the script does
- Queries the GitHub Contents API for the path site/content/docs.
- Recursively traverses subdirectories.
- Collects items where type == "file" and name ends with "_index.md" under the docs path.
- Excludes any paths that indicate Contribution guidelines, Privacy policy, or Release notes.
- Returns a list of blob URLs (the file URLs on GitHub) or any field you prefer (path, html_url, etc.).

Python script (plain text, to paste and run)
import os
import requests

REPO_OWNER = "kopia"
REPO_NAME = "kopia"
BASE_PATH = "site/content/docs"
REF = "master"  # or "main" if your default branch is main
GITHUB_API = "https://api.github.com"

# Optional: use a token to increase rate limits
TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
HEADERS = {}
if TOKEN:
    HEADERS["Authorization"] = f"token {TOKEN}"

# Exclusion patterns (adjust as needed)
EXCLUDE_PATTERNS = [
    "contributing",       # Contribution guidelines
    "privacy",              # Privacy policy
    "release-notes",        # Release notes
    "privacy-policy",
]

def is_excluded(path):
    lower = path.lower()
    for pat in EXCLUDE_PATTERNS:
        if pat in lower:
            return True
    return False

def list_docs_index_md_files(path):
    results = []
    url = f"{GITHUB_API}/repos/{REPO_OWNER}/{REPO_NAME}/contents/{path}?ref={REF}"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    items = resp.json()
    # items can be a dict if we hit a file directly; handle both
    if isinstance(items, dict):
        items = [items]

    for item in items:
        item_type = item.get("type")
        item_path = item.get("path", "")
        if is_excluded(item_path):
            continue
        if item_type == "dir":
            results.extend(list_docs_index_md_files(item_path))
        elif item_type == "file":
            name = item.get("name", "")
            if name.endswith("_index.md"):
                # Collect the blob URL (the URL to view the file on GitHub)
                blob_url = item.get("html_url") or item.get("url")
                # If html_url exists, that's the human-friendly page URL
                results.append(blob_url)
    return results

def main():
    docs_indexes = list_docs_index_md_files(BASE_PATH)
    # Optional: print as a simple list
    for u in docs_indexes:
        print(u)

if __name__ == "__main__":
    main()

How to run
- Save as get_docs_indexes_api.py
- Run: python get_docs_indexes_api.py
- If you want to limit output, you can pipe to head, or modify the script to write to a file.

Notes
- The script uses the GitHub Contents API at /repos/{owner}/{repo}/contents/{path}. It naturally recurses into subdirectories, but only within site/content/docs because BASE_PATH is set to that value.
- It returns the HTML URL (html_url) for each _index.md file, which looks like https://github.com/kopia/kopia/blob/master/site/content/docs/…/_index.md. If you prefer the raw file URL, you can use the "download_url" field (item.get("download_url")) instead.
- If your repo switched default branches (master -> main), set REF accordingly.

Option B: HTML parsing approach (fallback, with BeautifulSoup)
If you must parse the actual HTML pages on GitHub instead of the API, you can fetch the tree page at https://github.com/kopia/kopia/tree/master/site/content/docs and recursively follow subdirectories by parsing the page links. This is more brittle because GitHub’s HTML structure can change and pagination may appear for large trees.

What the HTML approach does
- Requests the HTML page for the docs directory.
- Uses BeautifulSoup to find links to files under this path whose URLs end with _index.md.
- For each linked directory, recurses into its page and repeats.
- Applies the same exclusion rules as above.

Python sketch (plain text)
import os
import requests
from bs4 import BeautifulSoup

START_URL = "https://github.com/kopia/kopia/tree/master/site/content/docs"

EXCLUDE_PATTERNS = [
    "contributing",
    "privacy",
    "release-notes",
    "privacy-policy",
]

def is_excluded_path(path):
    lower = path.lower()
    return any(p in lower for p in EXCLUDE_PATTERNS)

def extract_index_md_urls(page_url):
    r = requests.get(page_url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    urls = []
    for a in soup.find_all("a"):
        href = a.get("href", "")
        if not href:
            continue
        full = "https://github.com" + href
        # Heuristic: look for files ending with _index.md
        if href.endswith("_index.md"):
            if not is_excluded_path(href):
                urls.append(full)
        # If this is a directory link, it will look like /tree/...; you may choose to recurse
        if "/tree/master/site/content/docs" in href:
            dir_url = "https://github.com" + href
            if not is_excluded_path(href):
                urls.extend(extract_index_md_urls(dir_url))
    return urls

def main():
    urls = extract_index_md_urls(START_URL)
    for u in urls:
        print(u)

if __name__ == "__main__":
    main()

Notes on the HTML approach
- You’ll need BeautifulSoup installed (pip install beautifulsoup4 requests).
- This method is more fragile because it depends on GitHub’s HTML layout and may require handling pagination or dynamic loading.
- The API approach is generally preferred for reliability and speed.

Filtering examples
- If the repository adds new sections (e.g., a dedicated “contributing” section with _index.md), the exclusion patterns will skip any path containing contributing, privacy, or release-notes (case-insensitive). You can adjust EXCLUDE_PATTERNS to be more or less strict as needed.

What to do with the results
- The script prints the blob HTML URLs for each _index.md file. If you want to fetch the actual content, you can use the download_url or fetch the blob content via the GitHub API (GET https://api.github.com/repos/kopia/kopia/git/blobs/{sha}) or simply download_raw via download_url.

Assumptions and caveats
- The script targets the master branch as shown in the URL you provided. If the repo’s default branch is main or another branch, switch REF accordingly.
- The exclusion rules rely on path names containing certain keywords. If the project renames those sections, you may need to update the patterns.
- This approach returns file-level URLs to _index.md. If you instead want the site’s generated page URLs (e.g., the public docs pages), you’d need to map each _index.md file to its corresponding site URL (which Hugo usually builds from the file’s location) but that mapping isn’t directly available from the GitHub repo alone without building the site or inspecting the repository’s front matter and routing rules.

If you’d like, tell me whether you prefer:
- strictly GitHub blob URLs (as returned by the API), or
- the public URL path you would use after generating the site (which may require known routing rules), and I can tailor the code to return those exact URLs.

In [16]:
my_urls = get_urls()

In [17]:
print(my_urls)

['https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Actions/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Architecture/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Caching/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Compatibility/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Compression/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Consistency/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/ECC/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Encryption/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Kopiaignore/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advanced/Logging/_index.md', 'https://github.com/kopia/kopia/blob/master/site/content/docs/Advan

In [18]:
print(len(my_urls))

30


In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming

In [ ]:
# Get Llama 3.2 to answer